# 04 Feature Engineering

## Objective
Create ML-ready features directly in this notebook without using src modules.

## Features Built
- days_to_expiry: inventory perishability risk
- stock_velocity: stock depletion speed
- rolling_sales_average: short-term sales trend
- customer_lifetime_value: total customer spend
- purchase_frequency: customer visit intensity
- symptom_frequency: repeated symptom pattern signal
- inventory_demand_features: daily demand-driver table for forecasting (calendar + lag + momentum + interactions)

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

candidate_roots = [Path.cwd(), Path.cwd().parent]
project_root = next((p for p in candidate_roots if (p / 'data').exists()), None)
if project_root is None:
    raise FileNotFoundError('Could not locate data folder. Run from project root or notebooks folder.')

data_dir = project_root / 'data'
feature_dir = data_dir / 'features'
feature_dir.mkdir(parents=True, exist_ok=True)

print(f'Project Root: {project_root}')
print(f'Data Directory: {data_dir}')

Project Root: D:\projects\ai-ml-projects\PharmaEase_correct\pharma_ease_ai
Data Directory: D:\projects\ai-ml-projects\PharmaEase_correct\pharma_ease_ai\data


In [2]:
inventory = pd.read_csv(data_dir / 'inventory.csv')
sales = pd.read_csv(data_dir / 'sales.csv')
prescriptions = pd.read_csv(data_dir / 'prescriptions.csv')

print('Loaded datasets:')
print({
    'inventory': inventory.shape,
    'sales': sales.shape,
    'prescriptions': prescriptions.shape,
})

Loaded datasets:
{'inventory': (1500, 5), 'sales': (2000, 6), 'prescriptions': (1300, 4)}


In [3]:
def add_inventory_features(inventory_df: pd.DataFrame) -> pd.DataFrame:
    df = inventory_df.copy()
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    df['expiry_date'] = pd.to_datetime(df['expiry_date'], errors='coerce')

    df['days_to_expiry'] = (df['expiry_date'] - df['date']).dt.days.clip(lower=0)
    df['stock_velocity'] = (df['usage_rate'] / df['quantity'].replace(0, 1)).round(4)
    return df


def add_sales_features(sales_df: pd.DataFrame) -> pd.DataFrame:
    df = sales_df.copy()
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    df['revenue'] = df['quantity'] * df['price']

    daily_revenue = (
        df.groupby('date', as_index=False)['revenue']
        .sum()
        .sort_values('date')
        .reset_index(drop=True)
    )
    daily_revenue['rolling_sales_average'] = (
        daily_revenue['revenue'].rolling(window=7, min_periods=1).mean()
    )

    customer_agg = df.groupby('customer_id', as_index=False).agg(
        customer_lifetime_value=('revenue', 'sum'),
        purchase_frequency=('transaction_id', 'count'),
    )

    df = df.merge(customer_agg, on='customer_id', how='left')
    df = df.merge(daily_revenue[['date', 'rolling_sales_average']], on='date', how='left')
    return df


def add_prescription_features(prescriptions_df: pd.DataFrame) -> pd.DataFrame:
    df = prescriptions_df.copy()
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    df['symptom_frequency'] = df['symptoms'].map(df['symptoms'].value_counts())
    return df


def build_inventory_demand_features(
    inventory_df: pd.DataFrame, sales_df: pd.DataFrame, prescriptions_df: pd.DataFrame
) -> pd.DataFrame:
    inv = inventory_df.copy()
    sal = sales_df.copy()
    pre = prescriptions_df.copy()

    inv['date'] = pd.to_datetime(inv['date'], errors='coerce')
    sal['date'] = pd.to_datetime(sal['date'], errors='coerce')
    pre['date'] = pd.to_datetime(pre['date'], errors='coerce')

    inv_daily = inv.groupby('date', as_index=False).agg(
        usage_rate=('usage_rate', 'sum'),
        stock_on_hand=('quantity', 'sum'),
    )
    sal['revenue'] = sal['quantity'] * sal['price']
    sales_daily = sal.groupby('date', as_index=False).agg(
        sales_qty=('quantity', 'sum'),
        sales_txn=('transaction_id', 'count'),
        sales_revenue=('revenue', 'sum'),
    )
    pres_daily = pre.groupby('date', as_index=False).agg(
        prescription_count=('patient_id', 'count'),
        patient_count=('patient_id', 'nunique'),
    )

    all_dates = pd.date_range(
        min(inv_daily['date'].min(), sales_daily['date'].min(), pres_daily['date'].min()),
        max(inv_daily['date'].max(), sales_daily['date'].max(), pres_daily['date'].max()),
        freq='D',
    )
    base = pd.DataFrame({'date': all_dates})

    demand = base.merge(inv_daily, on='date', how='left')
    demand = demand.merge(sales_daily, on='date', how='left')
    demand = demand.merge(pres_daily, on='date', how='left')

    demand[['sales_qty', 'sales_txn', 'sales_revenue', 'prescription_count', 'patient_count']] = (
        demand[['sales_qty', 'sales_txn', 'sales_revenue', 'prescription_count', 'patient_count']].fillna(0)
    )
    demand['stock_on_hand'] = demand['stock_on_hand'].ffill().bfill()
    demand['usage_rate'] = demand['usage_rate'].fillna(0)

    for col in ['sales_qty', 'sales_txn', 'sales_revenue', 'prescription_count', 'patient_count', 'stock_on_hand']:
        demand[f'{col}_lag1'] = demand[col].shift(1).fillna(0)

    demand['dow'] = demand['date'].dt.dayofweek
    demand['month'] = demand['date'].dt.month
    demand['is_weekend'] = (demand['dow'] >= 5).astype(int)
    demand['is_month_end'] = demand['date'].dt.is_month_end.astype(int)

    for lag in [1, 2, 3, 7, 14, 21, 28]:
        demand[f'usage_lag_{lag}'] = demand['usage_rate'].shift(lag)

    demand['usage_roll_mean_7'] = demand['usage_rate'].shift(1).rolling(7).mean()
    demand['usage_roll_std_7'] = demand['usage_rate'].shift(1).rolling(7).std()
    demand['usage_roll_mean_14'] = demand['usage_rate'].shift(1).rolling(14).mean()
    demand['usage_roll_std_14'] = demand['usage_rate'].shift(1).rolling(14).std()
    demand['usage_ewm_7'] = demand['usage_rate'].shift(1).ewm(span=7, adjust=False).mean()

    demand['usage_x_stock'] = demand['usage_rate'] * demand['stock_on_hand']
    demand['usage_x_prescription'] = demand['usage_rate'] * demand['prescription_count_lag1']
    demand['demand_pressure'] = demand['prescription_count_lag1'] / (demand['stock_on_hand_lag1'] + 1)

    return demand.fillna(0)


def build_inventory_drug_daily_features(inventory_df: pd.DataFrame) -> pd.DataFrame:
    df = inventory_df.copy()
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    df['expiry_date'] = pd.to_datetime(df['expiry_date'], errors='coerce')
    df['days_to_expiry'] = (df['expiry_date'] - df['date']).dt.days.clip(lower=0)

    per_drug = df.groupby(['date', 'drug_name'], as_index=False).agg(
        usage_rate=('usage_rate', 'sum'),
        quantity=('quantity', 'sum'),
        days_to_expiry=('days_to_expiry', 'mean'),
    )
    per_drug['usage_share'] = per_drug['usage_rate'] / per_drug.groupby('date')['usage_rate'].transform('sum').replace(0, np.nan)
    per_drug['usage_share'] = per_drug['usage_share'].fillna(0)

    daily = per_drug.groupby('date', as_index=False).agg(
        total_usage_rate=('usage_rate', 'sum'),
        total_quantity=('quantity', 'sum'),
        drug_count=('drug_name', 'nunique'),
        mean_drug_usage=('usage_rate', 'mean'),
        std_drug_usage=('usage_rate', 'std'),
        max_drug_usage=('usage_rate', 'max'),
        mean_drug_quantity=('quantity', 'mean'),
        std_drug_quantity=('quantity', 'std'),
        max_drug_quantity=('quantity', 'max'),
        mean_days_to_expiry=('days_to_expiry', 'mean'),
    )

    concentration = per_drug.groupby('date')['usage_share'].apply(lambda s: float((s ** 2).sum())).reset_index(name='usage_concentration_hhi')
    top_share = per_drug.groupby('date')['usage_share'].max().reset_index(name='top_drug_share')
    daily = daily.merge(concentration, on='date', how='left').merge(top_share, on='date', how='left')

    all_dates = pd.date_range(daily['date'].min(), daily['date'].max(), freq='D')
    daily = pd.DataFrame({'date': all_dates}).merge(daily, on='date', how='left').fillna(0)

    daily['dow'] = daily['date'].dt.dayofweek
    daily['month'] = daily['date'].dt.month
    daily['is_weekend'] = (daily['dow'] >= 5).astype(int)
    daily['is_month_end'] = daily['date'].dt.is_month_end.astype(int)
    daily['is_quarter_end'] = daily['date'].dt.is_quarter_end.astype(int)

    for lag in [1, 2, 3, 7, 14, 21, 28]:
        daily[f'total_usage_lag_{lag}'] = daily['total_usage_rate'].shift(lag)
        daily[f'drug_count_lag_{lag}'] = daily['drug_count'].shift(lag)

    daily['total_usage_roll_mean_7'] = daily['total_usage_rate'].shift(1).rolling(7).mean()
    daily['total_usage_roll_std_7'] = daily['total_usage_rate'].shift(1).rolling(7).std()
    daily['total_usage_roll_mean_14'] = daily['total_usage_rate'].shift(1).rolling(14).mean()
    daily['total_usage_roll_std_14'] = daily['total_usage_rate'].shift(1).rolling(14).std()
    daily['total_usage_ewm_7'] = daily['total_usage_rate'].shift(1).ewm(span=7, adjust=False).mean()
    daily['concentration_roll_7'] = daily['usage_concentration_hhi'].shift(1).rolling(7).mean()
    daily['top_share_roll_7'] = daily['top_drug_share'].shift(1).rolling(7).mean()

    daily['usage_delta_7'] = daily['total_usage_rate'] - daily['total_usage_lag_7']
    daily['usage_delta_14'] = daily['total_usage_rate'] - daily['total_usage_lag_14']
    daily['drug_diversity'] = 1 / (daily['usage_concentration_hhi'].replace(0, np.nan))
    daily['drug_diversity'] = daily['drug_diversity'].replace([np.inf, -np.inf], 0).fillna(0)

    return daily.fillna(0)

In [4]:
inventory_features = add_inventory_features(inventory)
sales_features = add_sales_features(sales)
prescription_features = add_prescription_features(prescriptions)
inventory_demand_features = build_inventory_demand_features(inventory, sales, prescriptions)
inventory_drug_daily_features = build_inventory_drug_daily_features(inventory)

print('Feature tables ready:')
print({
    'inventory_features': inventory_features.shape,
    'sales_features': sales_features.shape,
    'prescription_features': prescription_features.shape,
    'inventory_demand_features': inventory_demand_features.shape,
    'inventory_drug_daily_features': inventory_drug_daily_features.shape,
})

Feature tables ready:
{'inventory_features': (1500, 7), 'sales_features': (2000, 10), 'prescription_features': (1300, 5), 'inventory_demand_features': (1155, 33), 'inventory_drug_daily_features': (1152, 42)}


In [5]:
print('\nInventory feature sample:')
display(inventory_features[['drug_name', 'quantity', 'usage_rate', 'days_to_expiry', 'stock_velocity']].head())

print('\nSales feature sample:')
display(sales_features[['customer_id', 'revenue', 'customer_lifetime_value', 'purchase_frequency', 'rolling_sales_average']].head())

print('\nPrescription feature sample:')
display(prescription_features[['patient_id', 'symptoms', 'symptom_frequency']].head())

print('\nInventory demand feature sample:')
display(
    inventory_demand_features[[
        'date', 'usage_rate', 'stock_on_hand', 'sales_qty_lag1', 'prescription_count_lag1',
        'usage_lag_7', 'usage_roll_mean_7', 'demand_pressure'
    ]].head()
 )

print('\nInventory drug daily feature sample:')
display(
    inventory_drug_daily_features[[
        'date', 'total_usage_rate', 'drug_count', 'usage_concentration_hhi',
        'top_drug_share', 'total_usage_roll_mean_7', 'usage_delta_7'
    ]].head()
 )


Inventory feature sample:


,drug_name,quantity,usage_rate,days_to_expiry,stock_velocity
0,Aspirin,274,12.48,332,0.0455
1,Paracetamol,169,7.34,315,0.0434
2,Amoxicillin,80,3.26,1031,0.0407
3,Paracetamol,410,17.96,1046,0.0438
4,Losartan,436,18.47,903,0.0424



Sales feature sample:


,customer_id,revenue,customer_lifetime_value,purchase_frequency,rolling_sales_average
0,CUST-2261,659.28,659.28,1,1085.92
1,CUST-2461,151.72,151.72,1,1085.92
2,CUST-1151,274.92,882.50,3,1085.92
3,CUST-1197,844.48,844.48,1,1314.26
4,CUST-2437,698.12,750.59,2,1314.26



Prescription feature sample:


,patient_id,symptoms,symptom_frequency
0,PAT-81636,fever headache,142
1,PAT-88964,respiratory infection cough,142
2,PAT-67983,fever headache,142
3,PAT-27009,high cholesterol chest discomfort,134
4,PAT-47576,mild pain blood thinning,119



Inventory demand feature sample:


,date,usage_rate,stock_on_hand,sales_qty_lag1,prescription_count_lag1,usage_lag_7,usage_roll_mean_7,demand_pressure
0,2023-01-01,23.08,523.0,0.0,0.0,0.0,0.0,0.000000
1,2023-01-02,0.00,523.0,0.0,1.0,0.0,0.0,0.001908
2,2023-01-03,17.96,410.0,9.0,1.0,0.0,0.0,0.001908
3,2023-01-04,18.47,436.0,0.0,1.0,0.0,0.0,0.002433
4,2023-01-05,0.00,436.0,11.0,1.0,0.0,0.0,0.002288



Inventory drug daily feature sample:


,date,total_usage_rate,drug_count,usage_concentration_hhi,top_drug_share,total_usage_roll_mean_7,usage_delta_7
0,2023-01-01,23.08,3.0,0.413477,0.540728,0.0,0.0
1,2023-01-02,0.00,0.0,0.000000,0.000000,0.0,0.0
2,2023-01-03,17.96,1.0,1.000000,1.000000,0.0,0.0
3,2023-01-04,18.47,1.0,1.000000,1.000000,0.0,0.0
4,2023-01-05,0.00,0.0,0.000000,0.000000,0.0,0.0


In [6]:
inventory_features.to_csv(feature_dir / 'inventory_features.csv', index=False)
sales_features.to_csv(feature_dir / 'sales_features.csv', index=False)
prescription_features.to_csv(feature_dir / 'prescription_features.csv', index=False)
inventory_demand_features.to_csv(feature_dir / 'inventory_demand_features.csv', index=False)
inventory_drug_daily_features.to_csv(feature_dir / 'inventory_drug_daily_features.csv', index=False)

print('Saved feature files:')
print('- inventory_features.csv')
print('- sales_features.csv')
print('- prescription_features.csv')
print('- inventory_demand_features.csv')
print('- inventory_drug_daily_features.csv')

Saved feature files:
- inventory_features.csv
- sales_features.csv
- prescription_features.csv
- inventory_demand_features.csv
- inventory_drug_daily_features.csv


## Feature Importance

### Inventory
- days_to_expiry helps identify expiration risk and replenishment urgency.
- stock_velocity captures relative depletion speed for each item.

### Sales
- rolling_sales_average captures local demand trend for forecasting.
- customer_lifetime_value supports customer segmentation and retention analysis.
- purchase_frequency helps model customer engagement intensity.

### Prescriptions
- symptom_frequency adds prevalence context that can improve NLP-based recommendations.

### Inventory Demand
- inventory_demand_features provides daily calendar, lag, rolling, and interaction signals for forecasting.
- inventory_drug_daily_features adds per-drug concentration, diversity, and trend signals for stronger demand modeling.